<a href="https://colab.research.google.com/github/aditisankara/lucidrt/blob/feat%2Ffine-tuning/src/notebooks/lucidrt_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Load dataset

In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20"

In [ ]:
from datasets import load_dataset

raw = load_dataset("lighteval/med_paragraph_simplification")

print("splits + sizes:", {k: len(v) for k, v in raw.items()})
first = next(iter(raw.values()))
print("columns:", first.column_names)
print("\none example:")
print(first[0])

splits + sizes: {'train': 3568, 'validation': 411, 'test': 480}
columns: ['query', 'answer']

one example:
{'query': 'Two trials met the inclusion criteria. One compared 2% ketanserin ointment in polyethylene glycol (PEG) with PEG alone, used twice a day by 40 participants with arterial leg ulcers, for eight weeks or until healing, whichever was sooner. One compared topical application of blood-derived concentrated growth factor (CGF) with standard dressing (polyurethane film or foam); both applied weekly for six weeks by 61 participants with non-healing ulcers (venous, diabetic arterial, neuropathic, traumatic, or vasculitic). Both trials were small, reported results inadequately, and were of low methodological quality. Short follow-up times (six and eight weeks) meant it would be difficult to capture sufficient healing events to allow us to make comparisons between treatments. One trial demonstrated accelerated wound healing in the ketanserin group compared with the control group. In

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("vocab size:", tok.vocab_size, "| pad:", tok.pad_token, "| eos:", tok.eos_token)

vocab size: 151643 | pad: <|endoftext|> | eos: <|im_end|>


In [ ]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)

ex = raw["train"][0]
prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
full_text = prompt + ex["answer"] + tok.eos_token   # what the model will learn from: instruction+input, then the answer

enc = tok(full_text)

print("num tokens:", len(enc["input_ids"]))
print("first 20 ids:", enc["input_ids"][:20])
print("round-trip decode:", tok.decode(enc["input_ids"][:20]))

num tokens: 878
first 20 ids: [50, 70206, 279, 2701, 6457, 1467, 1119, 14396, 4128, 264, 8720, 646, 3535, 382, 52554, 1467, 510, 11613, 19080, 2270]
round-trip decode: Simplify the following medical text into plain language a patient can understand.

Medical text:
Two trials met


In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 512

class MedSimplifyDataset(Dataset):
    def __init__(self, split):
        self.data = split          # a HF split, e.g. raw["train"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # TODO(you): same idea as Cell 4, but:
        #   1. ex = self.data[idx]; build prompt + full_text (as in Cell 4)
        #   2. tokenize WITH truncation=True, max_length=MAX_LEN   <- so the 878-token ones fit
        #   3. return a dict of 1-D tensors:
        #        {"input_ids": <tensor>, "attention_mask": <tensor>}
        #      The tokenizer gives python LISTS; wrap each with torch.tensor(..., dtype=torch.long)
        ex = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(technical=ex["query"])
        full_text = prompt + ex["answer"] + tok.eos_token
        enc = tok(full_text, truncation=True, max_length=MAX_LEN)
        return {
            "input_ids" : torch.tensor(enc["input_ids"]),
            "attention_mask" : torch.tensor(enc["attention_mask"])
        }

# quick self-test:
ds = MedSimplifyDataset(raw["train"])
print("dataset size:", len(ds))
item = ds[0]
print("keys:", item.keys())
print("input_ids:", item["input_ids"].shape, item["input_ids"].dtype)

dataset size: 3568
keys: dict_keys(['input_ids', 'attention_mask'])
input_ids: torch.Size([512]) torch.int64


In [ ]:
from torch.utils.data import DataLoader

ds = MedSimplifyDataset(raw["train"])
bad_loader = DataLoader(ds, batch_size=4)   # no collate_fn yet

try:
    batch = next(iter(bad_loader))
    print("input_ids:", batch["input_ids"].shape)
except Exception as e:
    print("ERROR:", type(e).__name__, "-", str(e)[:160])

ERROR: RuntimeError - stack expects each tensor to be equal size, but got [512] at entry 0 and [456] at entry 1


In [ ]:
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_token_id: int

    def __call__(self, batch):
        # batch = list of dicts, each {"input_ids":[L_i], "attention_mask":[L_i]} with DIFFERENT L_i
        max_len = max(example["input_ids"].size(0) for example in batch)

        input_ids, masks = [], []
        for ex in batch:
            ids, mask = ex["input_ids"], ex["attention_mask"]
            pad_len = max_len - ids.size(0)          # how many pad slots this example needs

            # TODO: right-pad `ids` with self.pad_token_id, and `mask` with 0, each by pad_len.
            #   hint: torch.cat([ids,  torch.full((pad_len,), self.pad_token_id, dtype=torch.long)])
            #         torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])
            ids  = torch.cat([ids, torch.full((pad_len, ), self.pad_token_id, dtype=torch.long)])
            mask = torch.cat([mask, torch.zeros(pad_len, dtype=torch.long)])

            input_ids.append(ids)
            masks.append(mask)

        # TODO: stack the lists of 1-D tensors into 2-D [B, max_len] tensors (hint: torch.stack)
        return {"input_ids": torch.stack(input_ids), "attention_mask": torch.stack(masks)}


# test it with a real DataLoader:
collator = PadCollator(pad_token_id=tok.pad_token_id)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collator)
batch = next(iter(loader))
print("input_ids:", batch["input_ids"].shape)       # expect [4, max_len_in_batch]
print("attention_mask:", batch["attention_mask"].shape)
print("mask row 0 (1=real, 0=pad):", batch["attention_mask"][0].tolist()[-10:])

input_ids: torch.Size([4, 512])
attention_mask: torch.Size([4, 512])
mask row 0 (1=real, 0=pad): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
def make_loaders(batch_size=4):
    collator = PadCollator(pad_token_id=tok.pad_token_id)
    train_dl = DataLoader(MedSimplifyDataset(raw["train"]),
                          batch_size=batch_size, shuffle=True,  collate_fn=collator)
    val_dl   = DataLoader(MedSimplifyDataset(raw["validation"]),
                          batch_size=batch_size, shuffle=False, collate_fn=collator)
    return train_dl, val_dl

train_dl, val_dl = make_loaders()
print("train batches:", len(train_dl), "| val batches:", len(val_dl))
xb = next(iter(train_dl))
print("one batch:", xb["input_ids"].shape)

train batches: 892 | val batches: 103
one batch: torch.Size([4, 512])


# 2. Model fine-tuning

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision -> fits the T4, trains faster
    device_map="cuda",           # put it on the GPU
)
n_params = sum(p.numel() for p in model.parameters())
print(f"{model.config.model_type} loaded | {n_params/1e9:.2f}B params | device: {next(model.parameters()).device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

qwen2 loaded | 1.54B params | device: cuda:0


In [ ]:
batch = next(iter(train_dl))
batch = {k: v.to("cuda") for k, v in batch.items()}   # move the batch onto the GPU (model is there)

# TODO(you): call the model. Pass input_ids, attention_mask, and labels=batch["input_ids"].
#   A HF causal LM, given `labels`, computes the loss for you and returns an object.
out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["input_ids"])

print("loss:", out.loss.item())
print("logits shape:", out.logits.shape)   # [batch, seq_len, vocab_size]

loss: 2.071124315261841
logits shape: torch.Size([4, 512, 151936])


In [ ]:
!pip install -q peft

In [ ]:
!pip uninstall -y torchao

In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8,                  # adapter "rank" — bigger = more capacity + more params. 8 is a fine default.
    lora_alpha=16,        # a scaling factor on the adapters
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # inject adapters into the attention layers
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
from torch.amp import autocast, GradScaler

# memory savers (you OOM'd before — these prevent it):
model.gradient_checkpointing_enable()   # recompute instead of storing activations
model.enable_input_require_grads()
model.config.use_cache = False          # required alongside gradient checkpointing

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
scaler = GradScaler("cuda")             # scales fp16 gradients so they don't vanish to 0
model.train()                           # put model in training mode
print("ready to train")

ready to train


In [ ]:
def make_labels(input_ids, attention_mask):
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100     # -100 tells the loss to IGNORE padding positions
    return labels

MAX_STEPS = 60
step = 0
for batch in train_dl:
    batch = {k: v.to("cuda") for k, v in batch.items()}
    labels = make_labels(batch["input_ids"], batch["attention_mask"])

    # ---- THE HEARTBEAT (you write these, in this order) ----
    # 1. clear leftover gradients:                 optimizer.zero_grad()
    # 2. forward pass, inside the fp16 autocast:
    #        with autocast("cuda", dtype=torch.float16):
    #            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=labels)
    #            loss = out.loss
    # 3. backward pass (scaled for fp16):          scaler.scale(loss).backward()
    # 4. update weights:                           scaler.step(optimizer); scaler.update()
    # replace this with the 4 steps above
    optimizer.zero_grad()
    with autocast("cuda", dtype=torch.float16):
      out = model(input_ids=batch["input_ids"],
                  attention_mask=batch["attention_mask"],
                  labels=labels)
      loss = out.loss
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    step += 1
    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.3f}")
    if step >= MAX_STEPS:
        break
print("training done")

step  10 | loss 1.763
step  20 | loss 1.763
step  30 | loss 1.662
step  40 | loss 1.816
step  50 | loss 1.660
step  60 | loss 1.656
training done


In [ ]:
model.eval()                              # eval mode (turns off dropout etc.)
model.gradient_checkpointing_disable()
model.config.use_cache = True             # re-enable the KV cache for fast generation — Day 3 is ALL about this

val_ex = raw["validation"][0]
prompt = PROMPT_TEMPLATE.format(technical=val_ex["query"])   # note: prompt only, NO answer this time
inputs = tok(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():                     # no training here -> don't track gradients (faster, less memory)
    gen = model.generate(**inputs, max_new_tokens=200, do_sample=False)

new_tokens = gen[0][inputs["input_ids"].shape[1]:]   # slice off the prompt; keep only what the model generated
print("=== MODEL OUTPUT ===")
print(tok.decode(new_tokens, skip_special_tokens=True))
print("\n=== HUMAN REFERENCE ===")
print(val_ex["answer"][:500])

=== MODEL OUTPUT ===
Interferon alfa-2b has been used successfully in treating chronic hepatitis C. This review found evidence from 54 studies involving 6545 participants who received interferon alfa-2b compared with placebo. In these studies, interferon alfa-2b improved liver function tests and reduced the amount of virus in the blood. It also led to fewer complications such as bleeding and infection. There was some evidence that interferon alfa-2b may improve liver function in people with cirrhosis. However, there was insufficient evidence to determine whether it improves liver function in people without cirrhosis. Interferon alfa-2b appears to be safe when given at high doses over long periods of time. However, side effects are more likely to occur if the drug is given at higher doses or for longer periods of time. Interferon alfa-2b should not be used in people with severe liver disease because it could cause further damage to their

=== HUMAN REFERENCE ===
The goal of this systema

In [ ]:
import os
model.save_pretrained("lucidrt-lora")
tok.save_pretrained("lucidrt-lora")
print("saved:", os.listdir("lucidrt-lora"))
size_mb = sum(os.path.getsize(os.path.join("lucidrt-lora", f)) for f in os.listdir("lucidrt-lora")) / 1e6
print(f"adapter size: {size_mb:.1f} MB")   # tiny! just the adapters, not the 1.5B base

saved: ['adapter_config.json', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'README.md']
adapter size: 20.2 MB
